# Production Pipeline: SmartAgro Subsidy Scoring

Полный пайплайн от данных до итогового скорa субсидии (0-100)

```
ЭТАП 1: Загрузка
ЭТАП 2: Feature Engineering
ЭТАП 3: Обучение XGBoost модели
ЭТАП 4: Скоринг заявителя (ML)
ЭТАП 5: Compliance Check (Embeddings + Negation)
ЭТАП 6: Итоговый скор субсидии (ML + Documents)
```

In [ ]:
# ============================================================
# ЭТАП 1: Загрузка и подготовка данных
# ============================================================
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("=" * 65)
print("  ЭТАП 1: Загрузка данных")
print("=" * 65)

df = pd.read_csv("../data/data.csv", skiprows=3, header=0, low_memory=False)

df.columns = [
    "num", "date_received", "col3", "col4", "region",
    "agimat", "app_number", "direction", "subsidy_name",
    "status", "normative", "amount", "district",
]

df = df[df["num"] != "№ п/п"].copy()
df.dropna(how="all", inplace=True)

for col in ["region", "direction", "status", "subsidy_name", "district"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df["normative"] = pd.to_numeric(df["normative"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["num"] = pd.to_numeric(df["num"], errors="coerce")
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce", dayfirst=True)

df = df[df["amount"].notna() & (df["amount"] > 0)].copy()

print(f"Загружено строк: {len(df):,}")

In [ ]:
# ============================================================
# ЭТАП 2: Feature Engineering
# ============================================================
from sklearn.preprocessing import LabelEncoder

print("\n" + "=" * 65)
print("  ЭТАП 2: Feature Engineering")
print("=" * 65)

df["hour_submitted"] = df["date_received"].dt.hour.fillna(12)
df["day_of_week"] = df["date_received"].dt.dayofweek.fillna(1)
df["month_submitted"] = df["date_received"].dt.month.fillna(1)

df["livestock_count"] = (df["amount"] / df["normative"].replace(0, np.nan)).round(0)
df["livestock_count"] = df["livestock_count"].clip(lower=1).fillna(10)
df["log_amount"] = np.log1p(df["amount"])

DIRECTION_MAP = {
    "Субсидирование в скотоводстве": 0,
    "Субсидирование в овцеводстве": 1,
    "Субсидирование в коневодстве": 2,
    "Субсидирование в птицеводстве": 3,
    "Субсидирование в верблюдоводстве": 4,
    "Субсидирование в свиноводстве": 5,
}
df["direction_code"] = df["direction"].map(DIRECTION_MAP).fillna(6)

df["is_pedigree"] = df["subsidy_name"].str.contains("племен", case=False, na=False).astype(int)
df["is_producer"] = df["subsidy_name"].str.contains("производи|производит", case=False, na=False).astype(int)

n = len(df)
base_growth = (df["log_amount"] - df["log_amount"].mean()) / df["log_amount"].std() * 0.1
pedigree_bonus = df["is_pedigree"] * np.random.uniform(0.05, 0.15, n)
noise_growth = np.random.normal(0, 0.12, n)
df["gross_output_growth_yoy"] = (base_growth + pedigree_bonus + noise_growth).clip(-0.30, 0.80)

direction_land_factor = df["direction_code"].map({0: 3.0, 1: 2.5, 2: 2.0, 3: 0.3, 4: 4.0, 5: 0.5, 6: 2.0}).fillna(2.0)
df["land_to_livestock_ratio"] = (direction_land_factor * np.random.lognormal(0, 0.4, n)).clip(0.2, 10.0)

region_survival = {
    "Мангистауская область": 0.82, "Атырауская область": 0.83,
    "Западно-Казахстанская область": 0.85, "Жамбылская область": 0.87,
    "Алматинская область": 0.90, "Акмолинская область": 0.88,
}
base_survival = df["region"].map(region_survival).fillna(0.87)
df["historical_survival_rate"] = (base_survival + np.random.normal(0, 0.05, n) + (df["direction_code"] == 3) * 0.04).clip(0.50, 0.99)

estimated_revenue = df["livestock_count"] * direction_land_factor * np.random.uniform(50000, 200000, n)
df["subsidy_dependence_index"] = (df["amount"] / (estimated_revenue + df["amount"])).clip(0.0, 1.0)

df["veterinary_compliance"] = (np.random.beta(8, 2, n) + df["is_pedigree"] * 0.05).clip(0.0, 1.0)

probs = np.array([1 / (1 + y * 0.3) for y in range(25)])
probs = probs / probs.sum()
df["years_in_operation"] = np.random.choice(range(1, 26), n, p=probs).astype(float)

df["pedigree_ratio"] = np.where(df["is_pedigree"] == 1, np.random.beta(5, 2, n), np.random.beta(2, 5, n))
df["previous_subsidies_count"] = np.random.poisson(lam=3.5, size=n).clip(0, 15)
df["debt_load_ratio"] = np.random.lognormal(mean=0.3, sigma=0.7, size=n).clip(0.0, 5.0)

def norm(series):
    rng = series.max() - series.min()
    return (series - series.min()) / rng if rng > 0 else series * 0

debt_inverted = 1 - norm(df["debt_load_ratio"])
raw_score = (
    norm(df["gross_output_growth_yoy"]) * 25.0 +
    norm(df["pedigree_ratio"]) * 20.0 +
    norm(df["historical_survival_rate"]) * 15.0 +
    norm(df["veterinary_compliance"]) * 13.0 +
    norm(df["subsidy_dependence_index"].apply(lambda x: 1-x)) * 12.0 +
    debt_inverted * 10.0 +
    norm(df["land_to_livestock_ratio"]) * 5.0 +
    norm(df["years_in_operation"]) * 5.0
)
raw_norm = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())
df["historical_score"] = (raw_norm * 99 + 1).round(1)
df["historical_score"] = (df["historical_score"] + np.random.normal(0, 3, len(df))).clip(1, 100).round(1)

le_region = LabelEncoder()
le_direction = LabelEncoder()
df["region_encoded"] = le_region.fit_transform(df["region"].fillna("Неизвестно"))
df["direction_encoded"] = le_direction.fit_transform(df["direction"].fillna("Неизвестно"))

print(f"Создано фичей: 17")
print(f"Целевая переменная: среднее={df['historical_score'].mean():.1f}")

In [ ]:
# ============================================================
# ЭТАП 3: Загрузка готовой модели (инференс)
# ============================================================
import joblib
from pathlib import Path

print("\n" + "=" * 65)
print("  ЭТАП 3: Загрузка готовой модели (инференс)")
print("=" * 65)

ML_FEATURES = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count",
    "debt_load_ratio", "log_amount", "livestock_count",
    "direction_code", "is_pedigree", "is_producer",
    "hour_submitted", "month_submitted", "region_encoded",
]
TARGET = "historical_score"

X = df[ML_FEATURES].copy()
y = df[TARGET].copy()
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

# Загружаем ГОТОВУЮ модель
model = joblib.load("../models/xgb_scorer.joblib")
scaler = joblib.load("../models/scaler.joblib")

print(f"Модель загружена: xgb_scorer.joblib")
print(f"Scaler загружен: scaler.joblib")
print(f"Фичей: {len(ML_FEATURES)}")

In [ ]:
# ============================================================
# ЭТАП 4: Скоринг конкретного заявителя (ML Score)
# ============================================================
import shap

print("\n" + "=" * 65)
print("  ЭТАП 4: Скоринг заявителя (ML Score)")
print("=" * 65)

# Берём заявителя из данных
idx = 0
one_farmer = X.iloc[[idx]].copy()
true_score = y.iloc[idx]

one_farmer_scaled = scaler.transform(one_farmer)
ml_score = float(np.clip(model.predict(one_farmer_scaled)[0], 1, 100))

print(f"\nЗаявитель: {df.iloc[idx].get('agimat', 'Н/Д')}")
print(f"Регион: {df.iloc[idx].get('region', 'Н/Д')}")
print(f"Сумма заявки: {df.iloc[idx]['amount']:,.0f} тенге")
print(f"\nML Score: {ml_score:.1f} / 100")

if ml_score >= 80:
    ml_zone = "GREEN"
elif ml_score >= 50:
    ml_zone = "YELLOW"
else:
    ml_zone = "RED"
print(f"Зона: {ml_zone}")

In [ ]:
# ============================================================
# ЭТАП 5: Compliance Check (Embeddings + Negation)
# ============================================================
import re
from sentence_transformers import SentenceTransformer, util

print("\n" + "=" * 65)
print("  ЭТАП 5: Compliance Check (Embeddings + Negation)")
print("=" * 65)

# Загружаем модель embeddings
print("Загрузка модели embeddings...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Модель загружена")

# Слова-отрицания
NEGATION_WORDS = [
    "не", "нет", "ни", "без", "отсутствует", "отсутствуют",
    "не выдана", "не выдано", "не получена", "не получено",
    "не проведена", "не проведено", "не зарегистрирован",
    "не действует", "не предоставлен", "не предоставлена",
    "снята", "снят", "снято", "изъят", "изъята",
    "расторгнут", "аннулирован", "лишен", "отказано",
    "запрещен", "запрещена", "запрещено",
]

# Требования для проверки (из compliance_checker.py)
SUBSIDY_RULES = {
    "КРС_маточное": {
        "name": "Приобретение племенного маточного поголовья КРС",
        "requirements": [
            {"id": "R-01", "text": "Наличие учётного номера хозяйства", "keywords": ["учетный номер", "учётный номер", "номер хозяйства"], "critical": True, "source": "Приложение 2, п.1"},
            {"id": "R-02", "text": "Наличие земель сельскохозяйственного назначения", "keywords": ["земельн", "кадастр", "гектар", "сельскохозяйственного назначения", "пастбищ"], "critical": True, "source": "Приложение 2, п.2"},
            {"id": "R-03", "text": "Регистрация поголовья в ИСЖ и ИБСПР", "keywords": ["ИСЖ", "ИБСПР", "регистрация поголовья", "идентификация животных", "ушная бирка"], "critical": True, "source": "Приложение 2, п.3"},
            {"id": "R-04", "text": "Возраст приобретённого поголовья", "keywords": ["возраст", "месяц", "телк", "нетел", "племенное свидетельство"], "critical": True, "source": "Приложение 2, п.4"},
            {"id": "R-05", "text": "Обязательство по целевому использованию не менее 2 лет", "keywords": ["обязательств", "два года", "2 года", "целевое использование", "воспроизводств"], "critical": True, "source": "Приложение 2, п.5"},
            {"id": "R-06", "text": "Акт или ЭСФ на приобретение", "keywords": ["акт", "ЭСФ", "счет-фактура", "приобретение", "порода", "договор купли"], "critical": True, "source": "Приложение 3"},
            {"id": "R-07", "text": "Справка о ветеринарном благополучии", "keywords": ["ветеринар", "благополучи", "ветсправка", "карантин", "эпизоотич"], "critical": True, "source": "Приложение 3"},
            {"id": "R-08", "text": "Банковские реквизиты", "keywords": ["ИИК", "БИК", "КБе", "банковские реквизиты", "расчётный счёт", "банк"], "critical": True, "source": "Заявка, п.4"},
        ],
    }
}

def check_compliance_embeddings(documents_text, rules, threshold=0.45):
    sentences = [s.strip() for s in re.split(r'[.!?]+', documents_text) if len(s.strip()) > 10]
    
    results = []
    for req in rules["requirements"]:
        query = " ".join(req["keywords"])
        query_emb = embed_model.encode(query, convert_to_tensor=True)
        
        best_score = 0.0
        best_sentence = ""
        negation_found = False
        
        for sentence in sentences:
            s_lower = sentence.lower()
            if any(neg in s_lower for neg in NEGATION_WORDS):
                negation_found = True
                continue
            
            sent_emb = embed_model.encode(sentence, convert_to_tensor=True)
            score = util.cos_sim(query_emb, sent_emb).item()
            
            if score > best_score:
                best_score = score
                best_sentence = sentence
        
        if negation_found:
            status = "НЕ НАЙДЕНО"
            evidence = "Обнаружено отрицание"
        elif best_score >= threshold:
            status = "ВЫПОЛНЕНО"
            evidence = f"cosine={best_score:.3f}"
        else:
            status = "НЕ НАЙДЕНО"
            evidence = f"cosine={best_score:.3f}"
        
        results.append({
            "id": req["id"],
            "text": req["text"],
            "status": status,
            "evidence": evidence,
            "score": round(best_score, 3),
            "critical": req["critical"],
            "source": req["source"],
        })
    
    return results

# Тестовый документ заявителя
applicant_doc = """
Справка от акимата Алматинской области.
Учетный номер хозяйства: 87654321.
Земельный участок сельскохозяйственного назначения: 150 гектар, кадастровый номер 01-22-333.
Все животные зарегистрированы в информационной системе идентификации и биркования.
Договор купли-продажи племенного маточного поголовья от 15.01.2024.
Акт приема-передачи и счет-фактура №456 на сумму 5 200 000 тенге.
Ветеринарная справка: хозяйство благополучно по инфекционным заболеваниям.
Обязательство по целевому использованию на 2 года оформлено.
Банковские реквизиты: ИИК KZ1234567890, БИК KZKOKZKX, КБе 001.
"""

rules = SUBSIDY_RULES["КРС_маточное"]
compliance_results = check_compliance_embeddings(applicant_doc, rules)

print(f"\nДокументы заявителя:")
print(f"Тип субсидии: {rules['name']}")
print(f"\nРезультаты проверки:")

for r in compliance_results:
    emoji = "✅" if r["status"] == "ВЫПОЛНЕНО" else "❌"
    crit = " [КРИТ]" if r["critical"] else ""
    print(f"  {emoji} {r['id']}: {r['status']}{crit} ({r['evidence']})")

# Считаем compliance score
total = len(compliance_results)
done = sum(1 for r in compliance_results if r["status"] == "ВЫПОЛНЕНО")
critical_failures = sum(1 for r in compliance_results if r["status"] == "НЕ НАЙДЕНО" and r["critical"])

doc_score = (done / total) * 100 if total > 0 else 0
print(f"\nCompliance Score: {doc_score:.0f}% ({done}/{total} требований)")
print(f"Критических нарушений: {critical_failures}")

In [ ]:
# ============================================================
# ЭТАП 6: Итоговый скор субсидии (ML + Documents)
# ============================================================
print("\n" + "=" * 65)
print("  ЭТАП 6: Итоговый скор субсидии")
print("=" * 65)

# Веса компонентов
ML_WEIGHT = 0.70
DOC_WEIGHT = 0.30

# Compliance bonus: от -20 до +10
if critical_failures > 0:
    compliance_bonus = -15.0 + (doc_score / 100) * 5
elif doc_score >= 85:
    compliance_bonus = (doc_score - 85) / 15 * 8
elif doc_score >= 60:
    compliance_bonus = (doc_score - 60) / 25 * 10 - 5
else:
    compliance_bonus = -10.0

compliance_bonus = round(max(-20.0, min(10.0, compliance_bonus)), 1)

# Итоговый скор
final_score = ml_score * ML_WEIGHT + doc_score * DOC_WEIGHT + compliance_bonus
final_score = round(max(0, min(100, final_score)), 1)

# Зона
if final_score >= 80:
    final_zone = "GREEN"
    recommendation = "Строго рекомендовано к включению в шорт-лист"
elif final_score >= 50:
    final_zone = "YELLOW"
    recommendation = "Рекомендуется дополнительное рассмотрение комиссией"
else:
    final_zone = "RED"
    recommendation = "Не рекомендовано — выявлены существенные риски"

print(f"\n{'='*50}")
print(f"  ИТОГОВЫЙ СКОР СУБСИДИИ")
print(f"{'='*50}")
print(f"  ML Score:       {ml_score:.1f} / 100 (вес {ML_WEIGHT:.0%})")
print(f"  Doc Score:      {doc_score:.0f} / 100 (вес {DOC_WEIGHT:.0%})")
print(f"  Compliance:     {compliance_bonus:+.1f} баллов")
print(f"{'='*50}")
print(f"  FINAL SCORE:    {final_score:.1f} / 100")
print(f"  Зона:           {final_zone}")
print(f"{'='*50}")
print(f"  Рекомендация:   {recommendation}")
print(f"{'='*50}")

---

# Итоговая формула

```
Final Score = ML_Score * 0.70 + Doc_Score * 0.30 + Compliance_Bonus

Где:
  ML_Score       — предсказание XGBoost (0-100)
  Doc_Score      — % выполненных требований документов
  Compliance_Bonus — бонус/штраф от -20 до +10

Зоны:
  GREEN  (80-100) — рекомендовано
  YELLOW (50-79)  — требует рассмотрения
  RED    (0-49)   — не рекомендовано
```